In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error
import joblib
import os

In [2]:
df = pd.read_csv('dataset_clean.csv')

In [3]:
# Define the target commodity for this specific model
all_commodities = df['variant_nama'].unique()

In [4]:
predicted_prices_tomorrow = {}

In [5]:
# Define features (X) and target (y)
features = ['day', 'day_of_week', 'is_weekend', 'price_lag_1', 'price_lag_2', 'price_lag_3', 'rolling_mean_7d']
X = df[features]
y = df['harga']

In [6]:
MODEL_DIR = 'trained_models'
os.makedirs(MODEL_DIR, exist_ok=True)

In [7]:
for commodity in all_commodities:
    # 1. Filter data per commodity
    df_model = df[df['variant_nama'] == commodity].copy()
    df_model = df_model.sort_values('tanggal').reset_index(drop=True)
    
    # Skip if data is too small (e.g., less than 30 days of data)
    if len(df_model) < 30:
        print(f"WARNING: Skipping {commodity} due to insufficient data.")
        continue
        
    # 2. Define Features and Target
    features = ['day', 'day_of_week', 'is_weekend', 'price_lag_1', 'price_lag_2', 'price_lag_3', 'rolling_mean_7d']
    X = df_model[features]
    y = df_model['harga']
    
    # 3. Train-Test Split (80/20)
    split_index = int(len(df_model) * 0.8)
    X_train, X_test = X.iloc[:split_index], X.iloc[split_index:]
    y_train, y_test = y.iloc[:split_index], y.iloc[split_index:]
    
    # 4. Initialize and Train Model
    model = XGBRegressor(n_estimators=100, learning_rate=0.1, max_depth=5, random_state=42)
    model.fit(X_train, y_train)
    
    # 5. Evaluate
    predictions = model.predict(X_test)
    mape = mean_absolute_percentage_error(y_test, predictions) * 100

    # 6. Predict Tomorrow's Price (Using the most recent row of data)
    # Get the last row of features and reshape it for prediction
    latest_features = X.iloc[-1:].copy() 
    tomorrow_price_pred = model.predict(latest_features)[0]
    
    # Store the result for the LP model
    predicted_prices_tomorrow[commodity] = tomorrow_price_pred
    
    # 7. Save the Model to Disk
    safe_filename = commodity.replace(" ", "_").replace("/", "_")
    model_path = os.path.join(MODEL_DIR, f"xgb_{safe_filename}.joblib")
    joblib.dump(model, model_path)


VISUALIZATION (Actual vs Predicted)

In [8]:
validation_results = []

for commodity in all_commodities:
    safe_filename = commodity.replace(" ", "_").replace("/", "_")
    model_path = os.path.join(MODEL_DIR, f"xgb_{safe_filename}.joblib")
    
    # Proceed if the trained model exists on disk
    if os.path.exists(model_path):
        # Load model
        model = joblib.load(model_path)
        
        # Prepare test data matching the training split
        df_model = df[df['variant_nama'] == commodity].sort_values('tanggal').reset_index(drop=True)
        features = ['day', 'day_of_week', 'is_weekend', 'price_lag_1', 'price_lag_2', 'price_lag_3', 'rolling_mean_7d']
        
        X = df_model[features]
        y = df_model['harga']
        split_index = int(len(df_model) * 0.8)
        X_test, y_test = X.iloc[split_index:], y.iloc[split_index:]
        
        # Predict and calculate error metrics
        predictions = model.predict(X_test)
        rmse = np.sqrt(mean_squared_error(y_test, predictions))
        mape = mean_absolute_percentage_error(y_test, predictions) * 100
        
        # Extract the most influential feature
        importances = model.feature_importances_
        top_feature_index = np.argmax(importances)
        top_feature = features[top_feature_index]
        
        # Append metrics
        validation_results.append({
            'Commodity': commodity,
            'MAPE (%)': round(mape, 2),
            'RMSE (IDR)': round(rmse, 2),
            'Top_Feature': top_feature
        })

# Generate and sort the validation report by lowest error
df_report = pd.DataFrame(validation_results)
df_report = df_report.sort_values(by='MAPE (%)').reset_index(drop=True)

print("INFO: Batch validation complete.")
print("Top 5 performing models:")
print(df_report.head())

print("\nWARNING: Bottom 5 models with the highest variance:")
print(df_report.tail())

# Export report for downstream analysis
output_file = 'model_evaluation_report.csv'
df_report.to_csv(output_file, index=False)
print(f"\nINFO: Evaluation report successfully saved to {output_file}.")

INFO: Batch validation complete.
Top 5 performing models:
            Commodity  MAPE (%)  RMSE (IDR)      Top_Feature
0        Beras Medium      0.08       12.31  rolling_mean_7d
1           Minyakita      0.11       20.56      price_lag_1
2       Tepung Terigu      0.11       16.29  rolling_mean_7d
3       Beras Premium      0.12       24.09  rolling_mean_7d
4  Bawang Putih Honan      0.28      155.78      price_lag_1

                    Commodity  MAPE (%)  RMSE (IDR)  Top_Feature
11           Gula Pasir Curah      1.29      259.52  price_lag_1
12       Cabai Merah Keriting      1.32      604.11  price_lag_1
13          Cabai Merah Besar      1.93     1061.89  price_lag_1
14          Cabai Rawit Merah      2.31     2041.52  price_lag_1
15  Minyak Goreng Sawit Curah      2.72      584.39  price_lag_1

INFO: Evaluation report successfully saved to model_evaluation_report.csv.
